# ReviewGuard: Hybrid Fake Review Detection
**Combining Transformer Semantics and Reviewer Behavior**

This notebook orchestrates the training and evaluation of the ReviewGuard system on Google Colab using a GPU accelerator.

### Project Overview
- **Text Branch:** Fine-tuned RoBERTa-base.
- **Behavioral Branch:** Hand-crafted reviewer signals.
- **Fusion:** MLP classifier combining semantic and behavioral embeddings.

---

## Step 1: Environment Setup
First, we clone the repository and install all necessary Python dependencies.

In [ ]:
# 1. Clone the repository
!git clone https://github.com/RohanMukka/Combining-Transformer-Semantics-and-Reviewer-Behavior-for-Fake-Review-Detection-on-Yelp.git
%cd Combining-Transformer-Semantics-and-Reviewer-Behavior-for-Fake-Review-Detection-on-Yelp

# 2. Install requirements
!pip install -r requirements.txt

# 3. Verify GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Data Loading & Preprocessing
**Important:** If you are uploading `yelpzip.csv` manually, put it in the `data/raw/` folder on the left sidebar before running this cell.

In [ ]:
# 1. Clear old cache to ensure we use your uploaded file
!rm -f data/processed/yelpzip.parquet

# 2. Run Data Loader
!python -m src.data_loader --dataset yelpzip

# 3. Sanity Check (Verify if Real or Synthetic)
import pandas as pd
import os
path = 'data/processed/yelpzip.parquet'
if os.path.exists(path):
    df = pd.read_parquet(path)
    print(f"\nSUCCESS: Loaded {len(df):,} reviews.")
    if len(df) > 60000:
        print("Dataset Status: ✅ REAL YelpZIP detected.")
    else:
        print("Dataset Status: ⚠️ SYNTHETIC data still in use.")
else:
    print("Error: Processed file not found.")

# 4. Compute behavioral features
!python -m src.behavior_features --dataset yelpzip

## Step 3: Orchestrated Training
This runs the full pipeline: Baselines -> RoBERTa -> Fusion -> Visualization.

In [ ]:
# Run the full pipeline (including professional figure generation)
!python -m src.train_all --dataset yelpzip

# Optional: Run SHAP explainability
!python -m src.explainability --dataset yelpzip --n_samples 200

## Step 4: Visualizing Report Figures
This section displays the consolidated figures used in the project report.

In [ ]:
import os
from IPython.display import Image, display

print("Displaying consolidated comparison charts...")

figs = [
    'results/figures/model_comparison_summary.png',
    'results/figures/roc_all_models.png',
    'results/figures/training_loss_curves.png',
    'results/shap_plots/shap_branch_importance.png'
]

for fig_path in figs:
    if os.path.exists(fig_path):
        print(f"\n--- Displaying: {os.path.basename(fig_path)} ---")
        display(Image(fig_path))
    else:
        print(f"Skipping {fig_path} (not found)")

## Step 5: Interactive Prediction

In [ ]:
print("Training complete. Models are ready for inference.")
# Note: You can implement a custom prediction script using the saved models in models/fusion/"